# Explorando o lakehouse

Tour pelas camadas do lakehouse, com **uma verificação por célula** (✅/❌, esperado × encontrado) e um quadro-resumo no fim.
Por enquanto só a **bronze** (fase 1). Silver e gold entram nas próximas fases.

**Regra de leitura (F3):** toda tabela Delta é lida pelo `_delta_log` (`DeltaTable` → DuckDB, função `ler_delta`), nunca pelos `.parquet` da pasta.

In [ ]:
import os
from pathlib import Path

# O notebook roda de dentro de notebooks/; o código do projeto está na raiz.
if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd
from deltalake import DeltaTable
from src.comum import carregar_config, conectar_duckdb, caminho_tabela, ler_delta

cfg = carregar_config()
con = conectar_duckdb(cfg)
TABELAS = [t["tabela"] for t in cfg["bronze"]["tabelas"]]
RASTREIO = ["_arquivo_origem", "_data_carga", "_carregado_em"]

resultados = []

def verificar(nome, esperado, encontrado, ok=None):
    """Registra uma verificação e mostra ✅/❌ com o esperado e o encontrado."""
    ok = (esperado == encontrado) if ok is None else ok
    resultados.append({"verificação": nome, "esperado": esperado, "encontrado": encontrado, "ok": "✅" if ok else "❌"})
    print(f"{'✅' if ok else '❌'} {nome}\n   esperado:   {esperado}\n   encontrado: {encontrado}")

def ultima_carga(tabela):
    dt = DeltaTable(str(caminho_tabela(cfg, "bronze", tabela)))
    return max(p["_data_carga"] for p in dt.partitions())

def bronze(tabela, carga=None):
    return ler_delta(con, caminho_tabela(cfg, "bronze", tabela), carga=carga)

print(f"lake: {cfg['paths']['lake']}")

lake: C:\Dashboard_Lakehouse\lake


## Bronze

Os 7 CSVs da Dunnhumby como chegaram: **tudo texto**, com `_arquivo_origem`, `_data_carga` (partição, data local de São Paulo) e `_carregado_em`.
Cada carga é uma partição; rodar de novo no mesmo dia substitui só aquela partição.

### Controle de cargas
Uma linha por tabela por execução. Abaixo, a execução mais recente de cada tabela.

In [ ]:
controle = bronze("_controle_cargas")
ultimas = con.sql("""
    SELECT * FROM controle
    QUALIFY row_number() OVER (PARTITION BY tabela ORDER BY carregado_em DESC) = 1
    ORDER BY tabela
""").df()
ultimas

,tabela,arquivo,data_carga,carregado_em,linhas_lidas,linhas_gravadas,colunas_esperadas_ok,status,mensagem
0,campaign_desc,campaign_desc.csv,2026-10-28,2026-10-28 16:32:25.028676-03:00,30,30,True,sucesso,None
1,campaign_table,campaign_table.csv,2026-10-28,2026-10-28 16:32:24.871739-03:00,7208,7208,True,sucesso,None
2,coupon,coupon.csv,2026-10-28,2026-10-28 16:32:25.173663-03:00,124548,124548,True,sucesso,None
3,coupon_redempt,coupon_redempt.csv,2026-10-28,2026-10-28 16:32:25.952967-03:00,2318,2318,True,sucesso,None
4,hh_demographic,hh_demographic.csv,2026-10-28,2026-10-28 16:32:24.642972-03:00,801,801,True,sucesso,None
5,product,product.csv,2026-10-28,2026-10-28 16:32:23.999695-03:00,92353,92353,True,sucesso,None
6,transaction_data,transaction_data.csv,2026-10-28,2026-10-28 16:32:03.219490-03:00,2595732,2595732,True,sucesso,None


### 1. Última carga das 7 tabelas com `sucesso`

In [ ]:
sucesso = sorted(ultimas.loc[ultimas["status"] == "sucesso", "tabela"])
verificar("última carga com sucesso", f"{len(TABELAS)} de {len(TABELAS)}", f"{len(sucesso)} de {len(TABELAS)}")

✅ última carga com sucesso
   esperado:   7 de 7
   encontrado: 7 de 7


### 2. Contagem de cada tabela = `linhas_gravadas` no controle
Contagem feita **pela tabela Delta**, na partição da última carga, contra o que a ingestão registrou.

In [ ]:
linhas = []
for t in TABELAS:
    carga = ultima_carga(t)
    contagem = bronze(t, carga).aggregate("count(*)").fetchall()[0][0]
    registro = ultimas.loc[ultimas["tabela"] == t].iloc[0]
    linhas.append({"tabela": t, "carga": carga, "contagem_delta": contagem,
                   "linhas_gravadas_controle": int(registro["linhas_gravadas"]),
                   "carga_do_controle": registro["data_carga"]})
contagens = pd.DataFrame(linhas)
display(contagens)
iguais = ((contagens["contagem_delta"] == contagens["linhas_gravadas_controle"])
          & (contagens["carga"] == contagens["carga_do_controle"])).sum()
verificar("contagem Delta = linhas_gravadas do controle", f"{len(TABELAS)} de {len(TABELAS)} tabelas", f"{iguais} de {len(TABELAS)} tabelas")

,tabela,carga,contagem_delta,linhas_gravadas_controle,carga_do_controle
0,transaction_data,2026-10-28,2595732,2595732,2026-10-28
1,product,2026-10-28,92353,92353,2026-10-28
2,hh_demographic,2026-10-28,801,801,2026-10-28
3,campaign_table,2026-10-28,7208,7208,2026-10-28
4,campaign_desc,2026-10-28,30,30,2026-10-28
5,coupon,2026-10-28,124548,124548,2026-10-28
6,coupon_redempt,2026-10-28,2318,2318,2026-10-28


✅ contagem Delta = linhas_gravadas do controle
   esperado:   7 de 7 tabelas
   encontrado: 7 de 7 tabelas


### 3. Colunas presentes = colunas esperadas do `config.yaml` (+ as 3 de rastreio)

In [ ]:
divergentes = []
for item in cfg["bronze"]["tabelas"]:
    presentes = [c for c in bronze(item["tabela"]).columns]
    if presentes != item["colunas"] + RASTREIO:
        divergentes.append(item["tabela"])
verificar("colunas = config + rastreio", "nenhuma tabela divergente", ", ".join(divergentes) or "nenhuma tabela divergente")

✅ colunas = config + rastreio
   esperado:   nenhuma tabela divergente
   encontrado: nenhuma tabela divergente


### 4. Colunas de dado 100% texto

In [ ]:
tipos = []
for t in TABELAS:
    for coluna, tipo in zip(bronze(t).columns, bronze(t).types):
        if coluna not in RASTREIO:
            tipos.append({"tabela": t, "coluna": coluna, "tipo": str(tipo)})
tipos = pd.DataFrame(tipos)
display(tipos.groupby("tipo").size().rename("colunas").to_frame())
verificar("colunas de dado como texto", f"{len(tipos)} de {len(tipos)} (100%)",
          f"{(tipos['tipo'] == 'VARCHAR').sum()} de {len(tipos)} ({(tipos['tipo'] == 'VARCHAR').mean():.0%})")

,colunas
tipo,
VARCHAR,41


✅ colunas de dado como texto
   esperado:   41 de 41 (100%)
   encontrado: 41 de 41 (100%)


### 5. Rastreio sem nulos (`_arquivo_origem`, `_data_carga`, `_carregado_em`), na tabela inteira

In [ ]:
nulos = []
for t in TABELAS:
    soma = " + ".join(f"count(*) FILTER (WHERE {c} IS NULL)" for c in RASTREIO)
    nulos.append({"tabela": t, "nulos_rastreio": bronze(t).aggregate(soma).fetchall()[0][0]})
nulos = pd.DataFrame(nulos)
display(nulos)
verificar("nulos nas colunas de rastreio", 0, int(nulos["nulos_rastreio"].sum()))

,tabela,nulos_rastreio
0,transaction_data,0
1,product,0
2,hh_demographic,0
3,campaign_table,0
4,campaign_desc,0
5,coupon,0
6,coupon_redempt,0


✅ nulos nas colunas de rastreio
   esperado:   0
   encontrado: 0


### 6. Gabarito
Números do projeto de RAG (relatório de qualidade): `transaction_data` com **2.595.732** linhas brutas e `product` com **92.353** produtos.

In [ ]:
n = contagens.set_index("tabela")["contagem_delta"]
verificar("transaction_data = gabarito", f"{2_595_732:,}".replace(",", "."), f"{n['transaction_data']:,}".replace(",", "."))

✅ transaction_data = gabarito
   esperado:   2.595.732
   encontrado: 2.595.732


In [ ]:
verificar("product = gabarito", f"{92_353:,}".replace(",", "."), f"{n['product']:,}".replace(",", "."))

✅ product = gabarito
   esperado:   92.353
   encontrado: 92.353


### 7. Versões Delta de cada tabela
Cada gravação gera uma versão nova; as antigas continuam disponíveis para *time travel* (`espiar.py ... --versao N`) até o `vacuum`.

In [ ]:
versoes = []
for t in TABELAS + ["_controle_cargas"]:
    hist = DeltaTable(str(caminho_tabela(cfg, "bronze", t))).history()
    ultima = max(hist, key=lambda h: h["version"])
    versoes.append({"tabela": t, "versao_atual": ultima["version"], "versoes": len(hist),
                    "ultima_operacao": ultima["operation"],
                    "predicado": (ultima.get("operationParameters") or {}).get("predicate")})
versoes = pd.DataFrame(versoes)
versoes

,tabela,versao_atual,versoes,ultima_operacao,predicado
0,transaction_data,1,2,WRITE,_data_carga = '2026-10-28'
1,product,1,2,WRITE,_data_carga = '2026-10-28'
2,hh_demographic,1,2,WRITE,_data_carga = '2026-10-28'
3,campaign_table,1,2,WRITE,_data_carga = '2026-10-28'
4,campaign_desc,1,2,WRITE,_data_carga = '2026-10-28'
5,coupon,1,2,WRITE,_data_carga = '2026-10-28'
6,coupon_redempt,1,2,WRITE,_data_carga = '2026-10-28'
7,_controle_cargas,13,14,WRITE,NaN


**Por que a regra F3 existe:** depois de duas cargas no mesmo dia, a pasta de `transaction_data` guarda os arquivos das duas versões.
Ler a pasta inteira com `*.parquet` contaria em dobro; a tabela Delta conta só os arquivos da versão atual.

In [ ]:
pasta = caminho_tabela(cfg, "bronze", "transaction_data")
carga = ultima_carga("transaction_data")
arquivos_na_pasta = len(list((pasta / f"_data_carga={carga}").glob("*.parquet")))
arquivos_da_versao = len(DeltaTable(str(pasta)).file_uris(file_pruning_predicate=[("_data_carga", "=", carga)]))
pela_pasta = con.sql(f"SELECT count(*) FROM read_parquet('{(pasta / f'_data_carga={carga}').as_posix()}/*.parquet')").fetchall()[0][0]
print(f"partição {carga}: {arquivos_na_pasta} arquivos .parquet na pasta, {arquivos_da_versao} na versão atual (pelo _delta_log)")
print("lendo a pasta direto (ERRADO, só para mostrar):", f"{pela_pasta:,}".replace(",", "."), "linhas")
verificar("leitura Delta não conta arquivos antigos",
          f"{2_595_732:,}".replace(",", "."),
          f"{bronze('transaction_data', carga).aggregate('count(*)').fetchall()[0][0]:,}".replace(",", "."))

partição 2026-10-28: 2 arquivos .parquet na pasta, 1 na versão atual (pelo _delta_log)
lendo a pasta direto (ERRADO, só para mostrar): 5.191.464 linhas


✅ leitura Delta não conta arquivos antigos
   esperado:   2.595.732
   encontrado: 2.595.732


## Quadro-resumo

In [ ]:
resumo = pd.DataFrame(resultados)[["ok", "verificação", "esperado", "encontrado"]]
print(f"{(resumo['ok'] == '✅').sum()} de {len(resumo)} verificações ✅")
resumo

8 de 8 verificações ✅


,ok,verificação,esperado,encontrado
0,✅,última carga com sucesso,7 de 7,7 de 7
1,✅,contagem Delta = linhas_gravadas do controle,7 de 7 tabelas,7 de 7 tabelas
2,✅,colunas = config + rastreio,nenhuma tabela divergente,nenhuma tabela divergente
3,✅,colunas de dado como texto,41 de 41 (100%),41 de 41 (100%)
4,✅,nulos nas colunas de rastreio,0,0
5,✅,transaction_data = gabarito,2.595.732,2.595.732
6,✅,product = gabarito,92.353,92.353
7,✅,leitura Delta não conta arquivos antigos,2.595.732,2.595.732
